> **Note:** This notebook requires a live OpenAI API key. Set `OPENAI_API_KEY` in your `.env` file before running. Smoke-run deferred — API key not available in CI.

# Stateful Chat with the Conversations API

The **Responses API** gives you two ways to keep multi-turn context without manually re-sending the whole message history on every call:

1. `client.conversations.create()` → a **durable, server-managed thread** you pass via `conversation=`.
2. `previous_response_id=` → **lightweight implicit chaining** off the prior response's id.

This notebook demonstrates both and contrasts them with manual history management.

In [ ]:
from openai import OpenAI

client = OpenAI()

# Create a durable conversation object. Its id persists across calls/sessions.
conversation = client.conversations.create()

first = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input=[{"role": "user", "content": "My name is Marcus. Remember it."}],
)
print(first.output_text)

In [ ]:
# Second turn — we do NOT re-send the first message. The server thread remembers it.
second = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input=[{"role": "user", "content": "What is my name?"}],
)
print(second.output_text)

In [ ]:
# Third turn — still no history re-send.
third = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    input=[{"role": "user", "content": "Spell my name backwards."}],
)
print(third.output_text)

## Contrast: the same exchange using `previous_response_id`

Instead of a persistent conversation object, we chain each call to the id of the previous response. Lighter weight, but the chain lives only as long as you hold the ids.

In [ ]:
r1 = client.responses.create(
    model="gpt-5.5",
    input="My name is Marcus. Remember it.",
)
print(r1.output_text)

r2 = client.responses.create(
    model="gpt-5.5",
    previous_response_id=r1.id,
    input=[{"role": "user", "content": "What is my name?"}],
)
print(r2.output_text)

## Comparison: three ways to carry context

| Mechanism | How state lives | Re-send history? | Best for |
|---|---|---|---|
| `conversation=<id>` | Durable server-managed thread object | No | Persistent chats you reattach to later (sessions, devices, jobs) |
| `previous_response_id=<id>` | Implicit chain off prior response | No | Lightweight short chains within one run |
| Manual `input=[...full history...]` | You hold every message in client code | Yes | Full control / custom truncation / portability |

## When to use each

- **`conversation=`** — pick this when you want a thread you can come back to: a chat app, a long-running assistant, anything spanning sessions.
- **`previous_response_id=`** — pick this for a quick chain of a few turns inside a single script where you already have the prior response in hand.
- **Manual history** — pick this when you need to shape exactly what the model sees (custom truncation, summarization, or provider portability).